In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')


# 1. Load the Dataset

In [2]:
df = pd.read_csv('../data/raw_data.csv')
df.shape


(5832, 26)

# 2. Data Cleaning

In [3]:
# Clean up column names by removing leading/trailing whitespace
df.columns = df.columns.str.strip()


In [4]:
# Drop duplicate rows
initial_rows = len(df)
df.drop_duplicates(inplace=True)
print(f"Removed {initial_rows - len(df)} duplicate rows.")
print(df.shape)


Removed 0 duplicate rows.
(5832, 26)


### Drop columns that are empty

In [5]:
# Find columns that contain only missing values
dropped_columns = df.columns[df.isna().all()].tolist()

# Drop them
df.dropna(axis=1, how='all', inplace=True)

print(f"Dropped {len(dropped_columns)} columns.")
print("Dropped columns:")
print(dropped_columns)

print(f"New shape: {df.shape}")


Dropped 0 columns.
Dropped columns:
[]
New shape: (5832, 26)


### Rename columns

In [6]:
df.columns

Index(['Timestamp', 'PV1_Current_A', 'PV2_Current_A', 'PV1_Voltage_V',
       'PV2_Voltage_V', 'PV1_Power_W', 'PV2_Power_W', 'PV_Total_Power_W',
       'Solar_Radiation_Wm2', 'Outdoor_Temp_C', 'Dew_Point_C', 'Wind_Speed_ms',
       'Wind_Dir_deg', 'Pressure_hPa', 'Humidity_pct', 'UV_Index',
       'Wind_Gust_ms', 'Rain_mm', 'AQI_US', 'PM25_ugm3', 'PM10_ugm3',
       'PM1_ugm3', 'Indoor_Temp_C', 'Indoor_Humidity_pct',
       'Indoor_Pressure_Pa', 'PV_imputed'],
      dtype='str')

In [7]:
column_mapping = {
        'PV_Total_Power_W': 'DC',
    }
df.rename(columns=column_mapping, inplace=True)
print("Renamed key columns for clarity.")

Renamed key columns for clarity.


In [8]:
df.head()

,Timestamp,PV1_Current_A,PV2_Current_A,PV1_Voltage_V,PV2_Voltage_V,PV1_Power_W,PV2_Power_W,DC,Solar_Radiation_Wm2,Outdoor_Temp_C,...,Wind_Gust_ms,Rain_mm,AQI_US,PM25_ugm3,PM10_ugm3,PM1_ugm3,Indoor_Temp_C,Indoor_Humidity_pct,Indoor_Pressure_Pa,PV_imputed
0,2026-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,10.192,...,1.5,0.0,52.550,10.500,18.850,8.167,8.758,75.500,101487.617,0
1,2026-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,10.008,...,1.0,0.0,46.517,8.433,14.950,6.550,8.487,76.700,101502.483,0
2,2026-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,9.875,...,1.0,0.0,46.383,8.433,15.017,6.417,8.333,78.700,101543.200,0
3,2026-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,9.367,...,1.0,0.0,51.800,9.617,17.317,7.567,7.933,81.017,101552.650,0
4,2026-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,8.925,...,1.5,0.0,52.767,9.983,17.033,8.033,7.430,82.733,101541.567,0


In [9]:
df.columns

Index(['Timestamp', 'PV1_Current_A', 'PV2_Current_A', 'PV1_Voltage_V',
       'PV2_Voltage_V', 'PV1_Power_W', 'PV2_Power_W', 'DC',
       'Solar_Radiation_Wm2', 'Outdoor_Temp_C', 'Dew_Point_C', 'Wind_Speed_ms',
       'Wind_Dir_deg', 'Pressure_hPa', 'Humidity_pct', 'UV_Index',
       'Wind_Gust_ms', 'Rain_mm', 'AQI_US', 'PM25_ugm3', 'PM10_ugm3',
       'PM1_ugm3', 'Indoor_Temp_C', 'Indoor_Humidity_pct',
       'Indoor_Pressure_Pa', 'PV_imputed'],
      dtype='str')

### Convert 'Hour' column to datetime objects

In [10]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')


### Replace '--' with NumPy's Not a Number (NaN)

In [11]:
# Replace placeholder characters ('--', etc.) with NumPy's Not a Number (NaN)
df.replace(['--', 'NA', 'NaN', 'nan'], np.nan, inplace=True)
print("Replaced placeholder text with NaN")


Replaced placeholder text with NaN


## Convert columns to numeric type

In [12]:
cols_to_convert = [
       'PV1_Current_A', 'PV2_Current_A', 'PV1_Voltage_V',
       'PV2_Voltage_V', 'PV1_Power_W', 'PV2_Power_W', 'DC',
       'Solar_Radiation_Wm2', 'Outdoor_Temp_C', 'Dew_Point_C', 'Wind_Speed_ms',
       'Wind_Dir_deg', 'Pressure_hPa', 'Humidity_pct', 'UV_Index',
       'Wind_Gust_ms', 'Rain_mm', 'AQI_US', 'PM25_ugm3', 'PM10_ugm3',
       'PM1_ugm3', 'Indoor_Temp_C', 'Indoor_Humidity_pct',
       'Indoor_Pressure_Pa', 'PV_imputed'
]

missing = [c for c in cols_to_convert if c not in df.columns]
if missing:
    print("Columns not found in df (check):", missing)

for col in cols_to_convert:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(',', '.', regex=False)
            .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Converted all feature columns to numeric type.")


Converted all feature columns to numeric type.


### Drop unnecessary columns

In [13]:
# Drop columns that are not needed for the analysis
cols_to_drop = [
 'PV_imputed',
 'Indoor_Temp_C', 'Indoor_Humidity_pct',
 'Indoor_Pressure_Pa'
]

existing_cols_to_drop = [c for c in cols_to_drop if c in df.columns]
missing_cols_to_drop = [c for c in cols_to_drop if c not in df.columns]

df.drop(columns=existing_cols_to_drop, inplace=True)

print(f"Dropped {len(existing_cols_to_drop)} columns:")
print(existing_cols_to_drop)
if missing_cols_to_drop:
    print("Not found in df (already absent):", missing_cols_to_drop)
print(f"New shape: {df.shape}")


Dropped 4 columns:
['PV_imputed', 'Indoor_Temp_C', 'Indoor_Humidity_pct', 'Indoor_Pressure_Pa']
New shape: (5832, 22)


In [14]:
df.columns

Index(['Timestamp', 'PV1_Current_A', 'PV2_Current_A', 'PV1_Voltage_V',
       'PV2_Voltage_V', 'PV1_Power_W', 'PV2_Power_W', 'DC',
       'Solar_Radiation_Wm2', 'Outdoor_Temp_C', 'Dew_Point_C', 'Wind_Speed_ms',
       'Wind_Dir_deg', 'Pressure_hPa', 'Humidity_pct', 'UV_Index',
       'Wind_Gust_ms', 'Rain_mm', 'AQI_US', 'PM25_ugm3', 'PM10_ugm3',
       'PM1_ugm3'],
      dtype='str')

In [15]:
df.shape

(5832, 22)

## Save cleaned data

In [16]:
df_cleaned = df
df_cleaned.to_csv("../data/cleaned_data.csv", index=False)
print(f"Saved cleaned data. Shape: {df_cleaned.shape}")


Saved cleaned data. Shape: (5832, 22)
